**Bloque 03. Manipulación de datos con pandas**

Revisión y práctica del *Python Data Science Handbook*, de Jake VanderPlas.
Este cuaderno contiene explicaciones y ejemplos propios en español.

El objetivo es pasar de una tabla con inconsistencias a un resumen de ventas estimadas, conservando el origen de las correcciones.

Requisitos: Python 3, Jupyter, NumPy y pandas. Ejecuta las celdas en orden desde un kernel nuevo.
Los datos de los ejemplos son sintéticos, salvo que se indique otra procedencia.
Cada bloque es independiente de los demás.

**1. Series, DataFrame e índices**

Una `Series` combina valores con etiquetas; un `DataFrame` organiza columnas
que pueden tener tipos diferentes. El índice participa en la selección y en
la alineación de operaciones. El CSV siguiente es sintético y está incluido
como texto para que el ejercicio funcione sin descargar archivos.

In [1]:
from io import StringIO
import numpy as np
import pandas as pd
from IPython.display import display

print('pandas:', pd.__version__)
csv_ventas = """id,fecha,tienda,producto,unidades,precio
1,2026-01-01, norte ,cafe,4,30
2,2026-01-01,SUR,te,6,20
3,2026-01-02,Norte,cacao,3,25
4,2026-01-02, sur ,cafe,5,30
5,2026-01-03,NORTE,te,?,20
6,2026-01-03,Sur,cacao,2,25
7,2026-01-08, norte ,cafe,7,30
8,2026-01-08,SUR,te,5,20
9,2026-01-09,Norte,cacao,4,25
10,2026-01-09, sur ,cafe,6,30
11,2026-01-10,NORTE,te,8,20
12,2026-01-10,Sur,cacao,3,25
"""
ventas_originales = pd.read_csv(StringIO(csv_ventas), skipinitialspace=True)
# Se añade deliberadamente un duplicado para practicar su detección.
ventas_sucias = pd.concat([ventas_originales, ventas_originales.iloc[[0]]],
                         ignore_index=True)
display(ventas_sucias.head())
display(ventas_sucias.dtypes.to_frame('tipo'))
print('Forma antes de limpiar:', ventas_sucias.shape)

pandas: 2.2.3


,id,fecha,tienda,producto,unidades,precio
0,1,2026-01-01,norte,cafe,4,30
1,2,2026-01-01,SUR,te,6,20
2,3,2026-01-02,Norte,cacao,3,25
3,4,2026-01-02,sur,cafe,5,30
4,5,2026-01-03,NORTE,te,?,20


,tipo
id,int64
fecha,object
tienda,object
producto,object
unidades,object
precio,int64


Forma antes de limpiar: (13, 6)


**2. Selección por etiqueta y por posición**

`.loc` selecciona etiquetas; `.iloc` selecciona posiciones. Asigno el ID como
índice para que la distinción sea visible. Primero retiro el duplicado exacto
y compruebo que cada ID corresponda a una sola operación. Una repetición con
valores contradictorios requeriría investigar el origen del registro.

In [2]:
print('Duplicados exactos:', ventas_sucias.duplicated().sum())
ventas = ventas_sucias.drop_duplicates().set_index('id').copy()
assert ventas.index.is_unique
display(ventas.loc[[2, 5], ['tienda', 'unidades']])
display(ventas.iloc[:2, :3])

Duplicados exactos: 1


,tienda,unidades
id,,
2,SUR,6
5,NORTE,?


,fecha,tienda,producto
id,,,
1,2026-01-01,norte,cafe
2,2026-01-01,SUR,te


**3. Alineación automática**

pandas alinea las operaciones por etiquetas, aunque los datos estén en otro
orden. Si una etiqueta no existe en ambos objetos, una suma ordinaria deja
un faltante. `fill_value=0` solo es apropiado cuando la ausencia se puede
interpretar como cero en esa operación.

In [3]:
enero = pd.Series({'NORTE': 10, 'SUR': 7})
febrero = pd.Series({'SUR': 9, 'CENTRO': 5})
display(pd.DataFrame({'enero': enero, 'febrero': febrero,
                     'suma_alineada': enero + febrero,
                     'suma_con_cero': enero.add(febrero, fill_value=0)}))

,enero,febrero,suma_alineada,suma_con_cero
CENTRO,NaN,5.0,NaN,5.0
NORTE,10.0,NaN,NaN,10.0
SUR,7.0,9.0,16.0,16.0


**4. Cadenas, fechas y valores faltantes**

Normalizo las tiendas quitando espacios y unificando mayúsculas. La conversión
numérica transforma `?` en un faltante detectable. En este ejemplo imputo las
unidades con la mediana del producto y conservo una marca de qué filas fueron
imputadas. Esa decisión sirve para practicar; necesita justificación del negocio
si se aplica a datos reales. El importe calculado con ella es una estimación.

In [4]:
ventas['tienda'] = ventas['tienda'].astype('string').str.strip().str.upper()
ventas['producto'] = ventas['producto'].astype('string').str.strip().str.lower()
ventas['fecha'] = pd.to_datetime(ventas['fecha'], format='%Y-%m-%d')
ventas['unidades'] = pd.to_numeric(ventas['unidades'], errors='coerce')
ventas['unidades_imputadas'] = ventas['unidades'].isna()
display(ventas.isna().sum().to_frame('faltantes'))

mediana_producto = ventas.groupby('producto')['unidades'].transform('median')
ventas['unidades_estimadas'] = ventas['unidades'].fillna(mediana_producto)
ventas['importe_estimado'] = ventas['unidades_estimadas'] * ventas['precio']
assert ventas['unidades_estimadas'].notna().all()
assert ventas['unidades_imputadas'].sum() == 1
display(ventas.loc[ventas['unidades_imputadas']])
print('Filas con unidades observadas:', ventas.dropna(subset=['unidades']).shape[0])

,faltantes
fecha,0
tienda,0
producto,0
unidades,1
precio,0
unidades_imputadas,0


,fecha,tienda,producto,unidades,precio,unidades_imputadas,unidades_estimadas,importe_estimado
id,,,,,,,,
5,2026-01-03,NORTE,te,NaN,20,True,6.0,120.0


Filas con unidades observadas: 11


**5. Operaciones por columna y filtros**

Las expresiones vectorizadas permiten crear variables y filtrar filas. Uso
`.loc` para seleccionar y asignar de forma explícita. `query()` ofrece otra
escritura para una condición; aquí se compara su resultado con la máscara.

In [5]:
seleccion = ventas.loc[(ventas['tienda'] == 'NORTE') &
                       (ventas['unidades_estimadas'] >= 5)]
consulta = ventas.query("tienda == 'NORTE' and unidades_estimadas >= 5",
                         engine='python')
pd.testing.assert_frame_equal(seleccion, consulta)
display(seleccion)
ventas['importe_con_descuento'] = ventas.eval('importe_estimado * 0.9')

,fecha,tienda,producto,unidades,precio,unidades_imputadas,unidades_estimadas,importe_estimado
id,,,,,,,,
5,2026-01-03,NORTE,te,NaN,20,True,6.0,120.0
7,2026-01-08,NORTE,cafe,7.0,30,False,7.0,210.0
11,2026-01-10,NORTE,te,8.0,20,False,8.0,160.0


**6. Concatenación y unión de tablas**

`concat` apila bloques con estructura compatible. `merge` incorpora columnas
buscando una clave común. El catálogo tiene una fila por producto, por lo que
`validate='many_to_one'` detecta duplicaciones inesperadas de esa clave.
`_merge` permite comprobar qué registros encontraron correspondencia.
En estos ejemplos se utiliza `pd.concat` para añadir filas.

In [6]:
primera_semana = ventas.loc[ventas['fecha'] < '2026-01-08']
segunda_semana = ventas.loc[ventas['fecha'] >= '2026-01-08']
reconstruida = pd.concat([primera_semana, segunda_semana]).sort_index()
pd.testing.assert_frame_equal(ventas.sort_index(), reconstruida)

catalogo = pd.DataFrame({'producto': ['cafe', 'te', 'cacao'],
                         'familia': ['cafeina', 'infusion', 'chocolate']})
enriquecida = ventas.reset_index().merge(
    catalogo, on='producto', how='left', validate='many_to_one', indicator=True,
)
assert len(enriquecida) == len(ventas)
assert enriquecida['_merge'].eq('both').all()
display(enriquecida.head())

,id,fecha,tienda,producto,unidades,precio,unidades_imputadas,unidades_estimadas,importe_estimado,importe_con_descuento,familia,_merge
0,1,2026-01-01,NORTE,cafe,4.0,30,False,4.0,120.0,108.0,cafeina,both
1,2,2026-01-01,SUR,te,6.0,20,False,6.0,120.0,108.0,infusion,both
2,3,2026-01-02,NORTE,cacao,3.0,25,False,3.0,75.0,67.5,chocolate,both
3,4,2026-01-02,SUR,cafe,5.0,30,False,5.0,150.0,135.0,cafeina,both
4,5,2026-01-03,NORTE,te,NaN,20,True,6.0,120.0,108.0,infusion,both


**7. Agrupación y tablas dinámicas**

`groupby` separa los registros por una clave y aplica un resumen. Muestro
juntos el importe estimado y cuántas operaciones contienen imputaciones.
La tabla dinámica cambia la presentación del mismo resumen para cruzar
tiendas y productos. Ambos cálculos deben conservar el total.

In [7]:
por_tienda = ventas.groupby('tienda').agg(
    operaciones=('producto', 'size'),
    unidades=('unidades_estimadas', 'sum'),
    importe_estimado=('importe_estimado', 'sum'),
    operaciones_imputadas=('unidades_imputadas', 'sum'),
)
pivote = ventas.pivot_table(index='tienda', columns='producto',
                             values='importe_estimado', aggfunc='sum',
                             fill_value=0, observed=True)
display(por_tienda)
display(pivote)
assert np.isclose(pivote.to_numpy().sum(), ventas['importe_estimado'].sum())

,operaciones,unidades,importe_estimado,operaciones_imputadas
tienda,,,,
NORTE,6,32.0,785.0,1
SUR,6,27.0,675.0,0


producto,cacao,cafe,te
tienda,,,
NORTE,175.0,330.0,280.0
SUR,125.0,330.0,220.0


**8. Índices jerárquicos**

Un `MultiIndex` identifica un dato con varias etiquetas. Agrupo por tienda
y producto y después uso `unstack` para pasar el nivel producto a columnas.
Es otra representación del cruce anterior, útil cuando hay varias dimensiones.

In [8]:
jerarquico = ventas.groupby(['tienda', 'producto'])['importe_estimado'].sum()
print('Nombres de los niveles:', jerarquico.index.names)
display(jerarquico.loc['NORTE'])
display(jerarquico.unstack('producto', fill_value=0))

Nombres de los niveles: ['tienda', 'producto']


producto
cacao    175.0
cafe     330.0
te       280.0
Name: importe_estimado, dtype: float64

producto,cacao,cafe,te
tienda,,,
NORTE,175.0,330.0,280.0
SUR,125.0,330.0,220.0


**9. Series de tiempo**

Después de agregar por fecha, `resample('D')` crea una frecuencia diaria.
En este CSV se supone que los días sin registros son días sin operaciones;
con esa suposición se pueden rellenar con cero. Si faltara la captura de
datos, correspondería conservar el faltante. La ventana móvil de tres días
resume la evolución local y no representa un pronóstico.

In [9]:
diario = (ventas.groupby('fecha')['importe_estimado'].sum()
          .resample('D').asfreq().fillna(0))
temporal = pd.DataFrame({'importe_estimado': diario,
                         'media_movil_3_dias': diario.rolling(3, min_periods=1).mean(),
                         'dia_semana': diario.index.day_name()})
display(temporal)
assert np.isclose(diario.sum(), ventas['importe_estimado'].sum())

,importe_estimado,media_movil_3_dias,dia_semana
fecha,,,
2026-01-01,240.0,240.000000,Thursday
2026-01-02,225.0,232.500000,Friday
2026-01-03,170.0,211.666667,Saturday
2026-01-04,0.0,131.666667,Sunday
2026-01-05,0.0,56.666667,Monday
2026-01-06,0.0,0.000000,Tuesday
2026-01-07,0.0,0.000000,Wednesday
2026-01-08,310.0,103.333333,Thursday
2026-01-09,280.0,196.666667,Friday


**10. Ejercicio resuelto y conclusión**

Pregunta: ¿qué tienda tiene el mayor importe estimado y cuánto de ese total
depende de la imputación? Se muestra tanto la suma con unidades observadas
como la estimación de las filas incompletas. El ranking depende de las
decisiones de limpieza y de las suposiciones señaladas.

In [10]:
observado = (ventas.loc[~ventas['unidades_imputadas']]
             .groupby('tienda')['importe_estimado'].sum())
aportacion_imputada = (ventas.loc[ventas['unidades_imputadas']]
                      .groupby('tienda')['importe_estimado'].sum())
respuesta = pd.DataFrame({'importe_observado': observado,
                           'importe_imputado': aportacion_imputada}).fillna(0)
respuesta['importe_estimado'] = respuesta.sum(axis=1)
display(respuesta.sort_values('importe_estimado', ascending=False))
print('Mayor importe estimado:', respuesta['importe_estimado'].idxmax())
assert np.isclose(respuesta['importe_estimado'].sum(), ventas['importe_estimado'].sum())

,importe_observado,importe_imputado,importe_estimado
tienda,,,
NORTE,665.0,120.0,785.0
SUR,675.0,0.0,675.0


Mayor importe estimado: NORTE


La trazabilidad de una limpieza requiere conservar los datos originales y
distinguir observación de estimación. Los índices explican la alineación;
las validaciones de unión evitan multiplicar filas; las agrupaciones y los
pivotes permiten comprobar que distintas vistas conservan los mismos totales.
`eval` y `query` facilitan escribir expresiones, pero un ejemplo pequeño no
demuestra una mejora de rendimiento.

**Fuentes y correspondencia con el libro**

Referencia indicada en la consigna: VanderPlas, J. (2022). *Python Data Science Handbook: Essential Tools for Working with Data*. O’Reilly Media.

El material efectivamente revisado es la carpeta `notebooks_v1` del enlace de la consigna. Su prefacio identifica el copyright de 2016; por eso la referencia de 2022 y estos materiales no se presentan como una misma edición. Las secciones siguientes permiten consultar el desarrollo original de los temas practicados aquí.

[03.00-Introduction-to-Pandas](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.00-Introduction-to-Pandas.ipynb).

[03.01-Introducing-Pandas-Objects](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.01-Introducing-Pandas-Objects.ipynb).

[03.02-Data-Indexing-and-Selection](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.02-Data-Indexing-and-Selection.ipynb).

[03.03-Operations-in-Pandas](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.03-Operations-in-Pandas.ipynb).

[03.04-Missing-Values](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.04-Missing-Values.ipynb).

[03.05-Hierarchical-Indexing](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.05-Hierarchical-Indexing.ipynb).

[03.06-Concat-And-Append](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.06-Concat-And-Append.ipynb).

[03.07-Merge-and-Join](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.07-Merge-and-Join.ipynb).

[03.08-Aggregation-and-Grouping](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.08-Aggregation-and-Grouping.ipynb).

[03.09-Pivot-Tables](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.09-Pivot-Tables.ipynb).

[03.10-Working-With-Strings](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.10-Working-With-Strings.ipynb).

[03.11-Working-with-Time-Series](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.11-Working-with-Time-Series.ipynb).

[03.12-Performance-Eval-and-Query](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.12-Performance-Eval-and-Query.ipynb).

[03.13-Further-Resources](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/03.13-Further-Resources.ipynb).

Documentación de las bibliotecas utilizada para adaptar los ejemplos:

[Concatenación con pandas](https://pandas.pydata.org/docs/reference/api/pandas.concat.html).

El texto original del libro declara licencia CC BY-NC-ND y su código, MIT. Las explicaciones y los ejercicios de este cuaderno son una elaboración nueva; las fuentes se conservan como referencia de estudio.